# Claude Agent SDK: custom tools (give Claude your own functions)

The built-in tools (Read, Bash, Grep ...) cover files and the shell. **Custom tools** let Claude call **your** Python functions: query your database, hit your internal API, run domain logic. You write a normal `async` function, describe it, and Claude decides **when** to call it.

Under the hood the SDK wraps your tools in an **in-process MCP server**: it runs inside your Python process, not as a separate program.

**What you will learn (and see running):**
1. The four parts of a tool: **name, description, input schema, handler**
2. Wrapping tools in a server and passing it as `mcp_servers`
3. The **fully qualified tool name** (`mcp__<server>__<tool>`) and `allowed_tools`
4. Three schema styles (dict of types, `Annotated`, full JSON Schema) and **optional parameters**
5. **Annotations** (`readOnlyHint`) and what they do (and do not) do
6. **Error handling**: raising vs returning `is_error`
7. Returning **images**
8. Controlling which tools exist: `tools`, `allowed_tools`, `disallowed_tools`

## The picture

```
  your Python function ──@tool──▶ SdkMcpTool ─┐
  your Python function ──@tool──▶ SdkMcpTool ─┼─▶ create_sdk_mcp_server("shop") ──▶ mcp_servers={"shop": server}
                                              ┘                                      allowed_tools=["mcp__shop__*"]

  Claude's loop:   think ─▶ call  mcp__shop__lookup_item({...}) ─▶ your handler runs ─▶ result text goes back to Claude
```

| Part | What it is | Who reads it |
|---|---|---|
| **name** | unique id, e.g. `lookup_item` | Claude (to call it) |
| **description** | what it does, when to use it | Claude (to decide) |
| **input schema** | arguments it accepts | Claude (to fill in) + the SDK (to validate) |
| **handler** | `async def f(args) -> {"content": [...]}` | your code |

## 0. Setup

Run these four cells first. They check Python, install the SDK, load your API key from a `.env` file, and create a private working folder for the demos.

In [ ]:
import sys
print(sys.version)
assert sys.version_info >= (3, 10), "The Agent SDK needs Python 3.10 or newer"

In [ ]:
%pip install -q claude-agent-sdk python-dotenv 

Create a file named `.env` next to this notebook containing one line (no quotes, no spaces around `=`):

```
ANTHROPIC_API_KEY=your-api-key
```

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

key = os.environ.get("ANTHROPIC_API_KEY", "")
if not key:
    raise RuntimeError(
        f"ANTHROPIC_API_KEY not found. Create a .env file in {Path.cwd()} with:\n"
        "ANTHROPIC_API_KEY=your-api-key"
    )
print("Loaded .env from:", env_path)
print("API key loaded:", key[:7] + "..." + key[-4:])

In [ ]:
from pathlib import Path

DEMO_DIR = Path("custom-tools-demo").resolve()
DEMO_DIR.mkdir(exist_ok=True)
print("Demo folder:", DEMO_DIR)

## 1. Imports and a small runner

`run()` sends one prompt and prints **every tool call, every tool result, and the final answer**, so you can watch Claude use your tools. It catches errors so one failure does not stop the notebook.

In [ ]:
import asyncio, json, base64, struct, zlib
from typing import Annotated, Any

from claude_agent_sdk import (
    query, tool, create_sdk_mcp_server, ToolAnnotations,
    ClaudeAgentOptions, AssistantMessage, UserMessage, ResultMessage,
    ToolUseBlock, ToolResultBlock, TextBlock,
)

async def run(prompt, **opts):
    """Run one query; print tool calls, tool results and the final answer."""
    opts.setdefault("cwd", str(DEMO_DIR))
    opts.setdefault("max_turns", 8)
    final, calls = "", []
    try:
        async for m in query(prompt=prompt, options=ClaudeAgentOptions(**opts)):
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, ToolUseBlock):
                        calls.append(b.name)
                        print(f"  [call]   {b.name}({json.dumps(b.input)})")
            elif isinstance(m, UserMessage) and isinstance(m.content, list):
                for b in m.content:
                    if isinstance(b, ToolResultBlock):
                        text = b.content if isinstance(b.content, str) else json.dumps(b.content)[:200]
                        flag = " (is_error)" if b.is_error else ""
                        print(f"  [result]{flag} {str(text)[:200]}")
            elif isinstance(m, ResultMessage):
                if m.subtype == "success":
                    final = m.result
                else:
                    print("  [ended with]", m.subtype)
    except Exception as error:
        print("  (query raised:", str(error)[:200], ")")
    print("ANSWER:", final)
    return final, calls

## 2. Your first tool: a unit converter

The `@tool` decorator takes `name`, `description`, `input_schema`. The function is the **handler**: it receives a dict of validated arguments and must return a dict with a `content` list.

Here we use **full JSON Schema** because we want an `enum` (a fixed set of allowed values). The dict-of-types shortcut cannot express enums.

In [ ]:
CONVERSIONS = {
    "length": {
        "kilometers_to_miles": lambda v: v * 0.621371,
        "miles_to_kilometers": lambda v: v * 1.60934,
        "meters_to_feet": lambda v: v * 3.28084,
        "feet_to_meters": lambda v: v * 0.3048,
    },
    "temperature": {
        "celsius_to_fahrenheit": lambda v: v * 9 / 5 + 32,
        "fahrenheit_to_celsius": lambda v: (v - 32) * 5 / 9,
    },
    "weight": {
        "kilograms_to_pounds": lambda v: v * 2.20462,
        "pounds_to_kilograms": lambda v: v * 0.453592,
    },
}

@tool(
    "convert_units",
    "Convert a numeric value from one unit to another (length, temperature or weight).",
    {
        "type": "object",
        "properties": {
            "unit_type": {"type": "string", "enum": ["length", "temperature", "weight"],
                          "description": "Category of unit"},
            "from_unit": {"type": "string", "description": "Unit to convert from, e.g. kilometers"},
            "to_unit": {"type": "string", "description": "Unit to convert to, e.g. miles"},
            "value": {"type": "number", "description": "The number to convert"},
        },
        "required": ["unit_type", "from_unit", "to_unit", "value"],
    },
)
async def convert_units(args: dict[str, Any]) -> dict[str, Any]:
    key = f"{args['from_unit']}_to_{args['to_unit']}"
    fn = CONVERSIONS.get(args["unit_type"], {}).get(key)
    if not fn:
        # Return the failure AS a result, flagged with is_error, so Claude can react.
        return {"content": [{"type": "text",
                             "text": f"Unsupported conversion: {args['from_unit']} to {args['to_unit']}"}],
                "is_error": True}
    return {"content": [{"type": "text",
                         "text": f"{args['value']} {args['from_unit']} = {fn(args['value']):.4f} {args['to_unit']}"}]}

print(type(convert_units).__name__, "| name =", convert_units.name)

### Test the handler directly (no AI, no API cost)

`@tool` returns an `SdkMcpTool`. Its `.handler` is your plain function, so you can unit-test it like any other code. **Do this first**: it is the fastest way to find bugs.

In [ ]:
# OFFLINE
print(await convert_units.handler({"unit_type": "length", "from_unit": "kilometers", "to_unit": "miles", "value": 100}))
print(await convert_units.handler({"unit_type": "length", "from_unit": "parsecs", "to_unit": "miles", "value": 1}))

### Wrap it in a server and let Claude use it

`create_sdk_mcp_server(name=..., tools=[...])` bundles tools. You hand it to `mcp_servers` under a **key of your choice**, and that key becomes part of every tool's name:

> `mcp__{server key}__{tool name}`  →  `mcp__converter__convert_units`

List that name in `allowed_tools` so Claude can run it **without a permission prompt**. (Unlisted tools go through the normal [permission flow](https://docs.claude.com), and in a script nobody is there to approve.)

In [ ]:
converter_server = create_sdk_mcp_server(name="converter", version="1.0.0", tools=[convert_units])

await run(
    "Convert 100 kilometers to miles, and also 72 degrees fahrenheit to celsius.",
    mcp_servers={"converter": converter_server},
    allowed_tools=["mcp__converter__convert_units"],
)

Look at the output: Claude made the calls itself, filled in `unit_type`, `from_unit`, ... from your sentence, got your handler's text back, and wrote the answer. It may also have called a `ToolSearch` first. That is **tool search** loading your tool's schema on demand (see the tool-search notebook).

Now an **unsupported** pair, to see `is_error` in action:

In [ ]:
await run(
    "Convert 3 parsecs to miles using the tool, and tell me exactly what the tool said.",
    mcp_servers={"converter": converter_server},
    allowed_tools=["mcp__converter__convert_units"],
)

## 3. Three ways to write an input schema, and optional parameters

| Style | Example | Notes |
|---|---|---|
| **Dict of types** | `{"latitude": float, "city": str}` | Shortest. **Every key is required.** |
| **Dict with `Annotated`** | `{"latitude": Annotated[float, "Latitude coordinate"]}` | Adds a description Claude sees. Still all required. |
| **Full JSON Schema** | `{"type": "object", "properties": {...}, "required": [...]}` | Enums, ranges, nested objects, **optional** fields. |

To make a parameter **optional**: use full JSON Schema, leave it out of `required`, and read it with `args.get("name", default)` in the handler.

We will build a tiny **inventory** server with one tool of each kind. The "database" is just a dict so everything runs offline.

In [ ]:
INVENTORY = {
    "apple":  {"price": 0.50, "stock": 120, "category": "fruit"},
    "banana": {"price": 0.25, "stock": 0,   "category": "fruit"},
    "carrot": {"price": 0.10, "stock": 300, "category": "vegetable"},
    "donut":  {"price": 1.75, "stock": 18,  "category": "bakery"},
}

# Style 2: dict with Annotated descriptions (all keys required)
@tool(
    "get_price",
    "Get the unit price in dollars of one inventory item.",
    {"item": Annotated[str, "Item name, lowercase, e.g. apple"]},
)
async def get_price(args):
    item = INVENTORY.get(args["item"].lower())
    if item is None:
        return {"content": [{"type": "text", "text": f"No such item: {args['item']}"}], "is_error": True}
    return {"content": [{"type": "text", "text": f"{args['item']}: ${item['price']:.2f}"}]}

# Style 3: full JSON Schema, with an OPTIONAL parameter ("category" is not in required)
@tool(
    "list_items",
    "List inventory items. Optionally filter by category (fruit, vegetable, bakery) and/or only in-stock items.",
    {
        "type": "object",
        "properties": {
            "category": {"type": "string", "description": "Optional category filter"},
            "in_stock_only": {"type": "boolean", "description": "Only items with stock > 0 (default false)"},
        },
        "required": [],
    },
)
async def list_items(args):
    category = args.get("category")                 # optional -> .get()
    in_stock_only = args.get("in_stock_only", False)
    rows = [
        f"{name} (${d['price']:.2f}, stock {d['stock']})"
        for name, d in INVENTORY.items()
        if (category is None or d["category"] == category)
        and (not in_stock_only or d["stock"] > 0)
    ]
    return {"content": [{"type": "text", "text": "\n".join(rows) or "(no matching items)"}]}

# Style 1: plain dict of types
@tool("restock_value", "Compute the total dollar value of `quantity` units of an item.",
      {"item": str, "quantity": int})
async def restock_value(args):
    d = INVENTORY.get(args["item"].lower())
    if d is None:
        return {"content": [{"type": "text", "text": f"No such item: {args['item']}"}], "is_error": True}
    return {"content": [{"type": "text", "text": f"${d['price'] * args['quantity']:.2f}"}]}

print(get_price.input_schema)
print(restock_value.input_schema)

In [ ]:
# OFFLINE
print(await list_items.handler({}))                                   # no optional args
print(await list_items.handler({"category": "fruit", "in_stock_only": True}))

### Several tools on one server, and the wildcard

With more than one tool you can list each name, or use `mcp__shop__*` to allow **every** tool on that server.

In [ ]:
shop_server = create_sdk_mcp_server(name="shop", tools=[get_price, list_items, restock_value])

await run(
    "Which fruit is in stock, and what would 40 apples cost in total?",
    mcp_servers={"shop": shop_server},
    allowed_tools=["mcp__shop__*"],
)

## 4. Annotations: hints, not enforcement

`ToolAnnotations` attaches metadata about how a tool behaves:

| Field | Default | Meaning |
|---|---|---|
| `readOnlyHint` | false | The tool does not modify anything. **Lets Claude run it in parallel with other read-only tools.** |
| `destructiveHint` | true | May do destructive updates (informational) |
| `idempotentHint` | false | Repeating the call has no extra effect (informational) |
| `openWorldHint` | true | Reaches systems outside your process (informational) |

**Annotations are labels, not locks.** A tool marked read-only can still write a file if its handler does. Keep the label honest.

In [ ]:
@tool(
    "get_stock",
    "Get the number of units in stock for one item.",
    {"item": str},
    annotations=ToolAnnotations(readOnlyHint=True),   # pure lookup, safe to batch
)
async def get_stock(args):
    d = INVENTORY.get(args["item"].lower())
    text = f"{args['item']}: {d['stock']} in stock" if d else f"No such item: {args['item']}"
    return {"content": [{"type": "text", "text": text}], **({} if d else {"is_error": True})}

print(get_stock.annotations)

In [ ]:
stock_server = create_sdk_mcp_server(name="stock", tools=[get_stock])

await run(
    "Check the stock of apple, banana, carrot and donut, all at once.",
    mcp_servers={"stock": stock_server},
    allowed_tools=["mcp__stock__get_stock"],
)

Because `get_stock` is read-only, Claude is **allowed** to run these calls in parallel with each other. Whether it actually batches them is Claude's decision, and the printed stream does not always make parallel calls obvious, so treat the hint as permission to batch rather than a guarantee.

## 5. Errors: raising vs returning `is_error`

A broken handler **never crashes the agent loop**. The SDK catches problems and gives Claude an error result. What differs is **what Claude reads**:

| What your handler does | What Claude sees |
|---|---|
| **Raises** an uncaught exception | The raw exception message, no context |
| **Returns** `"is_error": True` with your own text | **Your** message: you can say what failed and what to try instead |

Two tools that fail on purpose, one each way:

In [ ]:
@tool("fragile_lookup", "Look up a customer record by id.", {"customer_id": int})
async def fragile_lookup(args):
    raise KeyError(f"row {args['customer_id']}")          # uncaught -> raw message

@tool("friendly_lookup", "Look up a customer record by id (friendly errors).", {"customer_id": int})
async def friendly_lookup(args):
    try:
        raise KeyError(f"row {args['customer_id']}")
    except KeyError:
        return {"content": [{"type": "text",
                             "text": f"Customer {args['customer_id']} was not found. "
                                     "Ids start at 1000; ask the user to double-check the id."}],
                "is_error": True}

err_server = create_sdk_mcp_server(name="crm", tools=[fragile_lookup, friendly_lookup])

In [ ]:
print("=== raising ===")
await run("Use fragile_lookup for customer 42 and report what error text you got.",
          mcp_servers={"crm": err_server}, allowed_tools=["mcp__crm__fragile_lookup"])

print()
print("=== returning is_error ===")
await run("Use friendly_lookup for customer 42 and report what error text you got.",
          mcp_servers={"crm": err_server}, allowed_tools=["mcp__crm__friendly_lookup"])

Compare the `[result] (is_error)` lines. The friendly version gives Claude something it can act on (and relay to the user). **Rule of thumb:** catch the errors you can explain, return them with `is_error`, and let truly unexpected ones raise.

## 6. Returning images

A result's `content` list can hold `text`, `image`, `audio`, `resource` and `resource_link` blocks, mixed freely. An **image** block carries **base64 bytes inline** (there is no URL field), plus a `mimeType`. PNG, JPEG, GIF and WebP reach Claude as real visual input.

Rules to remember:
- `data` is raw base64 only, **no** `data:image/png;base64,` prefix
- `mimeType` is required
- To return an image from a URL: download it inside the handler and base64-encode the bytes
- **Python limits:** audio blocks are dropped (with a warning); binary `resource` blobs are dropped; `structuredContent` is not forwarded by the in-process server (use a standalone MCP server for that)

Below, a tool that draws a PNG (three coloured stripes) with pure Python and returns it. Then Claude is asked what colours it sees.

In [ ]:
def make_png(width, height, pixel):
    """Tiny pure-Python PNG writer. pixel(x, y) -> (r, g, b)."""
    raw = b""
    for y in range(height):
        raw += b"\x00" + b"".join(bytes(pixel(x, y)) for x in range(width))
    def chunk(tag, data):
        c = struct.pack(">I", len(data)) + tag + data
        return c + struct.pack(">I", zlib.crc32(tag + data) & 0xFFFFFFFF)
    return (b"\x89PNG\r\n\x1a\n"
            + chunk(b"IHDR", struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(raw))
            + chunk(b"IEND", b""))

STRIPES = [(220, 40, 40), (40, 180, 70), (40, 80, 220)]   # red, green, blue

@tool("draw_stripes", "Draw a small image of three vertical colour stripes and return it.", {})
async def draw_stripes(args):
    png = make_png(90, 60, lambda x, y: STRIPES[x // 30])
    return {"content": [
        {"type": "text", "text": "Here is the image."},
        {"type": "image", "data": base64.b64encode(png).decode("ascii"), "mimeType": "image/png"},
    ]}

art_server = create_sdk_mcp_server(name="art", tools=[draw_stripes])
(DEMO_DIR / "stripes.png").write_bytes(make_png(90, 60, lambda x, y: STRIPES[x // 30]))
print("Saved a copy you can open yourself:", DEMO_DIR / "stripes.png")

In [ ]:
await run(
    "Call draw_stripes, look at the image it returns, and tell me the colours from left to right.",
    mcp_servers={"art": art_server},
    allowed_tools=["mcp__art__draw_stripes"],
)

## 7. Controlling which tools exist and which are approved

Two **different layers** are easy to mix up:

| Layer | Question | Options |
|---|---|---|
| **Availability** | Is the tool in Claude's context at all? | `tools=[...]` (built-ins only), bare names in `disallowed_tools` |
| **Permission** | If Claude calls it, is it approved? | `allowed_tools`, scoped rules like `"Bash(rm *)"` in `disallowed_tools`, `permission_mode` |

| Option | Effect |
|---|---|
| `tools=["Read", "Grep"]` | Only these **built-ins** are available. MCP tools are unaffected. |
| `tools=[]` | **All built-ins removed.** Claude can use only your MCP tools. |
| `allowed_tools=[...]` | These run without a prompt. Others stay available but go through the permission flow. |
| `disallowed_tools=["Bash"]` | A bare name **removes** the tool from context. |

A sandbox where Claude can do **nothing except** call your tools:

In [ ]:
await run(
    "List the files in the current directory, then tell me the price of a donut.",
    mcp_servers={"shop": shop_server},
    tools=[],                                  # no built-ins: no Bash, no Read, nothing
    allowed_tools=["mcp__shop__*"],
)

Claude cannot list files (no built-ins exist) but can still use the shop tools. This is the safest pattern for an agent that should only touch your own API.

## 8. Summary and cheat sheet

```python
@tool("name", "what it does and when to use it", {"arg": str})        # schema: dict | Annotated | JSON Schema
async def handler(args):
    return {"content": [{"type": "text", "text": "..."}]}             # add "is_error": True on failure

server = create_sdk_mcp_server(name="myserver", tools=[handler])

ClaudeAgentOptions(
    mcp_servers={"myserver": server},
    allowed_tools=["mcp__myserver__*"],                               # mcp__<key>__<tool>
)
```

| I want to ... | Do this |
|---|---|
| Debug a tool cheaply | Call `my_tool.handler({...})` directly |
| Make a parameter optional | Full JSON Schema, omit from `required`, `args.get(...)` |
| Restrict a value to a set | `"enum": [...]` in JSON Schema |
| Allow parallel lookups | `annotations=ToolAnnotations(readOnlyHint=True)` |
| Give Claude a useful error | Return `"is_error": True` with your own message |
| Return a picture | `image` block, raw base64 + `mimeType` |
| Remove built-ins entirely | `tools=[]` |
| Scale to dozens of tools | See the **tool search** notebook |

**Habits worth keeping:** write descriptions for Claude, not for yourself (say *when* to use the tool); validate inputs in the handler; keep tool results short; never mark a tool read-only if it writes.

**Next:** connecting *external* MCP servers (`agent-sdk-mcp-servers-explained`) and handling large tool sets (`agent-sdk-tool-search-explained`).